1️⃣ Idea para el esquema JSON objetivo

In [1]:
propuesta_esquema ={
  "vehiculo": {
    "marca_vehiculo": "Toyota",
    "modelo": "Etios",
    "anio": 2013
  },
  "items": [
    {
      "descripcion_usuario": "pastillas de freno delanteras",
      "cantidad": 1
    },
    {
      "descripcion_usuario": "bomba de freno",
      "cantidad": 1
    }
  ],
  "notas": "algún comentario del usuario si hace falta"
}

2️⃣ Lo bajamos a Pydantic para usar con Groq + LangChain

In [2]:
from pydantic import BaseModel, Field
from typing import List, Optional

class VehiculoPedido(BaseModel):
    marca_vehiculo: Optional[str] = Field(
        None, 
        description="Marca del vehículo (Toyota, Renault, etc.)"
    )
    modelo: Optional[str] = Field(
        None, 
        description="Modelo del vehículo (Etios, Clio, etc.)"
    )
    anio: Optional[str] = Field(
        None,
        description="Año o rango de años del modelo (por ejemplo '2016' o '2016-2020')."
    )

class ItemPedido(BaseModel):
    descripcion_usuario: str = Field(..., description="Descripción del repuesto que permita buscarlo en el catálogo")
    cantidad: int = Field(1, description="Cantidad pedida de ese repuesto (por defecto es 1)")

class PedidoRepuestos(BaseModel):
    vehiculo: VehiculoPedido
    items: List[ItemPedido]
    notas: Optional[str] = Field(None, description="Comentarios generales sobre el pedido, como plazos o forma de entrega")


3️⃣ Prompt para que el LLM haga SOLO extracción → JSON

In [3]:
from langchain_core.prompts import ChatPromptTemplate

prompt_pedido = ChatPromptTemplate.from_messages([
    ("system",
     "Sos un asistente que recibe pedidos de repuestos escritos en lenguaje natural. "
     "Debés identificar la marca, modelo y año del vehículo (si aparece) y una lista de repuestos pedidos. "
     "La salida debe ajustarse exactamente al modelo Pydantic 'PedidoRepuestos'. "
     "No agregues campos que no están en el modelo. "
     "No inventes datos que no aparezcan en el texto, excepto cantidades (usar 1 si no se especifica). "
     "Extraé los datos de la forma más fiel posible."),
    
    ("human",
     "Texto del pedido:\n\n{texto_pedido}")
])


/home/nicolas/entornos/trabajo-final-modulo6_python3.11.13/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# Le decimos al LLM que queremos la salida como PedidoRepuestos
llm_struct = llm.with_structured_output(PedidoRepuestos)

def parsear_pedido(texto_pedido: str) -> PedidoRepuestos:
    try:
        mensaje = prompt_pedido.format_messages(texto_pedido=texto_pedido)
        return llm_struct.invoke(mensaje)
    except Exception as e:
        print("⚠️ Error al interpretar el pedido:", e)
        raise


4️⃣ Uso con with_structured_output (LangChain + Groq)

In [ ]:
texto = """
Necesito para un Toyota Etios 2016: pastillas de freno delanteras y una bomba de freno. También estoy necesitando un juego de amortiguadores traseros para un Renault Clio 2018. Por favor, entregarlo en el taller antes del viernes.
"""
## hasta acá no puede armar múltiples pedidos
pedido = parsear_pedido(texto)

print("Objeto Pydantic:")
print(pedido)

print("\nPedido como JSON:")
print(pedido.model_dump_json(indent=2))

Objeto Pydantic:
vehiculo=VehiculoPedido(marca_vehiculo='Toyota', modelo='Etios', anio='2016') items=[ItemPedido(descripcion_usuario='pastillas de freno delanteras', cantidad=1), ItemPedido(descripcion_usuario='bomba de freno', cantidad=1)] notas='Entregar antes del viernes'

Pedido como JSON:
{
  "vehiculo": {
    "marca_vehiculo": "Toyota",
    "modelo": "Etios",
    "anio": "2016"
  },
  "items": [
    {
      "descripcion_usuario": "pastillas de freno delanteras",
      "cantidad": 1
    },
    {
      "descripcion_usuario": "bomba de freno",
      "cantidad": 1
    }
  ],
  "notas": "Entregar antes del viernes"
}
